# Allies versus Lessons play study

Explore how a player behaves in recorded games. Change the parameters, run all cells, and read the tables. The same configuration drives `python -m manabot.study.allies_lessons`, so anything found here can be re-run from the command line.

Start Jupyter from the repository root: `uv run --extra notebook jupyter lab experiments/study/`

## Parameters

`OVERRIDES` replaces fields of the saved configuration. Set `RECORD = True` to play games; leave it `False` to read the record already in `OUT`.

In [ ]:
OUT = ".runs/study/allies-lessons"
CONFIG = None  # path to a study.json to start from, or None for defaults
OVERRIDES = {}  # for example {"mirror_deals": 100, "gw_allies": "-2 Badgermole Cub, +2 Plains"}
RECORD = False

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

from manabot.study import frames, measures as m, read_games
from manabot.study.allies_lessons import build_report, learn_choices
from manabot.study.allies_lessons.__main__ import run
from manabot.study.allies_lessons.config import StudyConfig
from manabot.study.allies_lessons.report import pairing

# Relative paths are read from the repository root.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
out = root / OUT
saved = out / "study.json"
source = root / CONFIG if CONFIG else saved if saved.exists() and not RECORD else None
config = (StudyConfig.load(source) if source else StudyConfig()).changed(**OVERRIDES)
config

In [ ]:
if RECORD:
    run(config, out)
found = [p for p in (out / "games.jsonl.gz", out / "games.jsonl") if p.exists()]
if not found:
    raise FileNotFoundError(f"No games recorded in {out}. Set RECORD = True, or point OUT at an existing run.")
games = read_games(found[0])
frames.games_frame(games).groupby(["player_0", "player_1", "end"]).size().rename("games").reset_index()

## Which deck wins, on the play and on the draw?

A mirror puts the same player in both seats, so these rates describe the decks and the starting seat.

In [ ]:
mirror = pairing(games, config.subject, config.subject)
frames.rates_frame(m.win_rates(mirror, config.subject), "deck / position")

In [ ]:
seats = frames.seats_frame(mirror)
seats[seats.finished].pivot_table(index="deck", columns="position", values="score", aggfunc=["mean", "count"])

## How does it play?

In [ ]:
played = [game for game in games if game.seats(config.subject)]
questions = {
    "played a land when offered": m.land_drops,
    "attacked with no blocker showing": m.open_attacks,
    "attacked with a blocker showing": m.contested_attacks,
    "blocked when offered": m.blocks,
    "cast a spell when castable": m.casting,
}
pd.concat(
    {
        player: frames.rates_frame({q: f(games, player) for q, f in questions.items()}, "behavior").set_index("behavior")
        for player in dict.fromkeys([config.subject, config.baseline])
    },
    axis=1,
)

In [ ]:
frames.rates_frame(m.land_drops_by_lands(played, config.subject), "lands in play")

In [ ]:
frames.rates_frame(m.held_back(played, config.subject), "creature").head(12)

## Ask your own question

`decisions_frame` has one row per decision. This example asks how often the subject passed priority on its own main phase with untapped lands and a castable spell among its offers.

In [ ]:
decisions = frames.decisions_frame(played)
mine = decisions[(decisions.player == config.subject) & decisions.own_turn]
main = mine[mine.step.str.contains("MAIN") & (mine.kind == "PRIORITY")]
main.groupby("deck").apply(lambda d: (d.chosen == "PRIORITY_PASS_PRIORITY").mean(), include_groups=False).rename("passed priority in a main phase")

In [ ]:
pd.Series(learn_choices(played, config.subject), name="Learn choices").sort_values(ascending=False)

## The report

The same page the command writes, rebuilt from the games loaded above.

In [ ]:
display(HTML(build_report(games, subject=config.subject, baseline=config.baseline, fragment=True)))